# Gradient-Boosted Magnitude Prediction with XGBoost

## What this notebook does
This notebook loops through every CSV file in a Google Drive input folder, removes highly correlated predictors, creates training, validation, and test subsets, tunes an XGBoost regression model with Optuna, evaluates it on held-out data, and saves one results table per dataset.

## Required inputs
- Google Colab with access to Google Drive.
- One or more CSV files stored directly in `input_folder`.
- Each CSV must contain the columns expected by the preprocessing section described below.

## Outputs
- One `<dataset>_XGB_results.csv` file per input CSV, containing test R-squared, MAE, RMSE, MAPE, and the best hyperparameters.
- Progress and training diagnostics printed in the notebook output.

## Settings a new user should review
- `input_folder`: change this to the Google Drive folder containing the input CSV files.
- `output_folder`: change this to the folder where result CSV files should be written.
- `df.drop("site_id", axis=1)`: edit or remove this line if your identifier column has a different name or is absent.
- `df = df[df.columns[1:]]`: edit or remove this line if the first remaining column should not be discarded.
- The correlation threshold (`0.9`), split proportions, random seed, and number of Optuna trials can be changed if required.

## Important data assumptions
- After `site_id` is removed, the script discards the first remaining column.
- The first column left after those removals is treated as the response variable; all later columns are predictors.
- The input folder should contain only CSV files that follow this same structure, because every CSV in the folder is processed.

Run the notebook from top to bottom in Google Colab. The progress messages explain what is happening and where outputs are written.

## 1. Install required packages
Run this cell once at the start of a fresh Colab session.

In [ ]:
# Install Packages
!pip install keras-tuner tensorflow pandas numpy scikit-learn shap
!pip install optuna --quiet

## 2. Import packages

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import optuna
#import shap
import os
import xgboost as xgb
from xgboost import DMatrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
from optuna.pruners import MedianPruner

## 3. Mount Google Drive and set directories
**Edit `input_folder` and `output_folder` in the next cell before running the notebook on a new system.**

In [ ]:
# Mount Google Drive so the notebook can read inputs and save outputs
drive.mount('/content/drive')

# User configuration: input and output directories
input_folder = '/content/drive/My Drive/machine_learning_colab/'
output_folder = '/content/drive/My Drive/machine_learning_colab/outputs_new/'
os.makedirs(output_folder, exist_ok = True)

## 4. Process each dataset, tune the model, and save results

In [ ]:
for file_name in os.listdir(input_folder):
    if file_name.endswith('.csv'):
        dataset_path = os.path.join(input_folder, file_name)
        dataset_name = file_name.replace('.csv', '')
        print(f"\nStarting XGBoost modelling for dataset: {dataset_name}")

        # Load dataset
        df = pd.read_csv(dataset_path)
        print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns from {file_name}.")
        df = df.drop('site_id', axis=1)
        df = df[df.columns[1:]]

        # Remove highly correlated features
        corr_matrix = df.iloc[:, 1:].corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
        df = df.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} predictors with absolute pairwise correlation above 0.90.")

        # Separate response and predictors
        Y = df.iloc[:, 0].values.reshape(-1, 1)
        X = df.iloc[:, 1:]
        feature_names = X.columns.tolist()

        # Remove rows with NaN in Y
        mask = ~np.isnan(Y).flatten()
        X = X[mask]
        Y = Y[mask]

        # Standardize predictors
        scaler_x = StandardScaler()
        X_scaled = scaler_x.fit_transform(X)

        # Split data: 60% train, 20% validation, 20% test
        X_train_val, X_test, Y_train_val, Y_test = train_test_split(
            X_scaled, Y, test_size=0.2, random_state=42)

        X_train, X_val, Y_train, Y_val = train_test_split(
            X_train_val, Y_train_val, test_size=0.25, random_state=42)  # 0.25 * 0.8 = 0.2

        # Convert to DMatrix for XGBoost
        dtrain = xgb.DMatrix(X_train, label=Y_train)
        dvalid = xgb.DMatrix(X_val, label=Y_val)
        dtest = xgb.DMatrix(X_test, label=Y_test)

        def objective(trial):
            params = {
                'objective': 'reg:squarederror',
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'learning_rate': trial.suggest_loguniform('learning_rate', 0.005, 0.3),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
                'subsample': trial.suggest_float('subsample', 0.4, 1.0),
                'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-5, 1e1),
                'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-5, 1e1),
            }

            evals_result = {}
            model = xgb.train(
                params,
                dtrain,
                num_boost_round=500,
                evals=[(dvalid, 'validation')],
                early_stopping_rounds=20,
                verbose_eval=False,
                evals_result=evals_result
            )

            # Get best validation RMSE from training history
            best_rmse = min(evals_result['validation']['rmse'])
            return best_rmse

        pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5, interval_steps=1)
        print("Starting Optuna optimization with 50 trials. This may take some time for larger datasets.")
        study = optuna.create_study(direction='minimize', pruner=pruner)
        study.optimize(objective, n_trials=50, show_progress_bar=True)

        best_params = study.best_trial.params
        print(f"Optimization finished. Best parameters: {best_params}")
        best_params['objective'] = 'reg:squarederror'

        # Train final model on combined train + validation data
        dtrain_final = xgb.DMatrix(X_train_val, label=Y_train_val)
        final_model = xgb.train(
            best_params,
            dtrain_final,
            num_boost_round=2000,
            evals=[(dtest, 'test')],
            early_stopping_rounds=20,
            verbose_eval=True
        )

        test_preds = final_model.predict(dtest)

        def evaluate(y_true, y_pred, label='Test'):
            test_r2 = r2_score(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mape = mean_absolute_percentage_error(y_true, y_pred)
            return test_r2, mae, rmse, mape

        test_r2, mae, rmse, mape = evaluate(Y_test, test_preds, label='Test')

        results_df = pd.DataFrame([{
            "Dataset": dataset_name,
            "Test_R2": test_r2,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape,
            "Best_Params": best_params
        }])

        output_file = os.path.join(output_folder, f"{dataset_name}_XGB_results.csv")
        results_df.to_csv(output_file, index=False)
        print(f"Finished {dataset_name}. Results were saved to: {output_file}")